# Notebook 5 / R5 - WasteNet-128K Teacher-Assistant KD

Notebook ini menjalankan smoke test reproducibility dan confirmation pilot R5 untuk WasteNet-128K dengan teacher-assistant Focus-RCNet dari R2.
Default notebook disetel ke confirmation Pilot 1: `T=4`, `alpha=0.05`, 60 epoch, tanpa early stopping dan tanpa evaluasi test.


## Rules

- Gunakan `split_manifest_seed_*.csv` dari R0.
- Student init wajib berasal dari checkpoint R3 WasteNet-128K CE final pada seed yang sama.
- Teacher assistant untuk preset KD wajib berasal dari R2 Focus-RCNet direct KD final pada seed yang sama.
- Pilot R5 hanya memakai validation set untuk memilih `temperature`, `alpha`, dan `stage2_lr`.
- Independent test set hanya dievaluasi ketika `RUN_PHASE = "final"` setelah konfigurasi R5 dibekukan.
- Jalankan dua smoke test identik (`R5_WORKFLOW_STAGE=smoke`, replika `01` dan `02`) sebelum full pilot.
- Smoke test hanya memeriksa reproducibility dan tidak dipakai untuk memilih konfigurasi.
- Setelah smoke test identik, jalankan tiga confirmation pilot 60 epoch tanpa early stopping.
- Confirmation queue: `ta_t4p0_a0p05_lr0p0005`, `ta_t2p0_a0p3_lr0p0005`, `ce_finetune_lr0p001`.
- Student dan assistant menerima augmentasi acak yang sama sebelum dibuat menjadi view 160 dan 380 piksel.
- Independent test set tetap tidak dievaluasi pada smoke test maupun pilot.


## R5 Pilot Presets

Gunakan `PILOT_PRESET` atau env var `R5_PILOT_PRESET` untuk menjalankan konfigurasi pilot lain tanpa edit manual.
Total pilot penuh: 2 CE fine-tune controls + 18 KD grid runs + 2 anchor runs = 22 runs.

Default confirmation Pilot 1 setelah kedua smoke test identik:

```python
WORKFLOW_STAGE = "pilot"
RUN_REPLICATE = "01"
PILOT_PRESET = "ta_t4p0_a0p05_lr0p0005"
```

Urutan eksekusi:

1. Smoke 1: `R5_WORKFLOW_STAGE=smoke`, `R5_RUN_REPLICATE=01`.
2. Smoke 2: `R5_WORKFLOW_STAGE=smoke`, `R5_RUN_REPLICATE=02`.
3. Pastikan history fingerprint dan prediction fingerprint kedua smoke identik.
4. Pilot KD-1: `R5_WORKFLOW_STAGE=pilot`, `R5_PILOT_PRESET=ta_t4p0_a0p05_lr0p0005`.
5. Pilot KD-2: `R5_WORKFLOW_STAGE=pilot`, `R5_PILOT_PRESET=ta_t2p0_a0p3_lr0p0005`.
6. Pilot CE: `R5_WORKFLOW_STAGE=pilot`, `R5_PILOT_PRESET=ce_finetune_lr0p001`.

Jangan menjalankan stage `final` sebelum konfigurasi R5 dibekukan.


In [ ]:
# ============================================================
# 1. Imports
# ============================================================

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import copy
import hashlib
import json
import os
import random
import time
import warnings

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
warnings.filterwarnings("ignore", category=UserWarning)

print("Imports ready")


In [ ]:
# ============================================================
# 2. Configuration
# ============================================================

class Config:
    EXPERIMENT_ID = "R5"
    EXPERIMENT_NAME = "R5 WasteNet-128K Teacher-Assistant KD"
    PLATFORM = "Kaggle Notebooks"
    FRAMEWORK = "PyTorch"
    USE_AMP = True
    MULTI_GPU = False
    DETERMINISTIC = True

    SEED = 42
    RUN_PHASE = "pilot"
    WORKFLOW_STAGE = "pilot"
    RUN_REPLICATE = "01"

    DATASET_NAME = "TrashNet"
    DATASET_DIR = Path(os.environ.get(
        "TRASHNET_DATASET_DIR",
        "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
    ))

    R0_DIR_CANDIDATES = [
        Path(os.environ.get("R0_DATA_PROTOCOL_DIR", "")),
        Path("/kaggle/input/notebook0-r0-final-data-protocol-setup/final_research/r0_data_protocol"),
        Path("/kaggle/input/notebook0-r0-final-data-protocol-setup/final_research"),
        Path("/kaggle/input/notebook0-r0-final-data-protocol-setup"),
        Path("/kaggle/input/notebook0_r0_final_data_protocol_setup/final_research/r0_data_protocol"),
        Path("/kaggle/input/notebook0_r0_final_data_protocol_setup/final_research"),
        Path("/kaggle/input/notebook0_r0_final_data_protocol_setup"),
        Path("/kaggle/input/r0-kaggle-final/r0_data_protocol"),
        Path("/kaggle/input/thesis-kd-trashnet/final_research_kd/r0_data_protocol"),
        Path("/kaggle/input/thesis-kd-trashnet/final_research_kd/runs/r0/final_research/r0_data_protocol"),
        Path("/kaggle/working/final_research/r0_data_protocol"),
        Path.cwd() / "final_research_kd" / "r0_data_protocol",
        Path.cwd() / "final_research_kd" / "runs" / "r0" / "final_research" / "r0_data_protocol",
        Path.cwd() / "runs" / "r0" / "final_research" / "r0_data_protocol",
    ]
    R2_ASSISTANT_ROOT_CANDIDATES = [
        Path(os.environ.get("R2_ASSISTANT_DIR", "")),
        Path("/kaggle/input/output-notebook2-r2-focus-rcnet-kd-teacher-assistant-selection/final_research_kd/runs/final/R2/direct_kd"),
        Path("/kaggle/input/notebook2-r2-focus-rcnet-kd-teacher-assistant-selection/final_research_kd/runs/final/R2/direct_kd"),
        Path("/kaggle/input/notebook2-r2-focus-rcnet-kd-teacher-assistant-selection/runs/final/R2/direct_kd"),
        Path("/kaggle/input/r2-final/final_research_kd/runs/final/R2/direct_kd"),
        Path("/kaggle/input/thesis-kd-trashnet/final_research_kd/runs/final/R2/direct_kd"),
        Path("/kaggle/working/final_research_kd/runs/final/R2/direct_kd"),
        Path.cwd() / "final_research_kd" / "runs" / "final" / "R2" / "direct_kd",
    ]
    R3_STUDENT_ROOT_CANDIDATES = [
        Path(os.environ.get("R3_STUDENT_DIR", "")),
        Path("/kaggle/input/output-notebook3-r3-wastenet-128k-ce-baseline/final_research_kd/runs/final/R3"),
        Path("/kaggle/input/notebook3-r3-wastenet-128k-ce-baseline/final_research_kd/runs/final/R3"),
        Path("/kaggle/input/notebook3-r3-wastenet-128k-ce-baseline/runs/final/R3"),
        Path("/kaggle/input/r3-final/final_research_kd/runs/final/R3"),
        Path("/kaggle/input/thesis-kd-trashnet/final_research_kd/runs/final/R3"),
        Path("/kaggle/working/final_research_kd/runs/final/R3"),
        Path.cwd() / "final_research_kd" / "runs" / "final" / "R3",
    ]

    DEFAULT_OUTPUT_ROOT = (
        Path("/kaggle/working/final_research_kd/runs")
        if Path("/kaggle/working").exists()
        else Path.cwd() / "final_research_kd" / "runs"
    )
    OUTPUT_ROOT = Path(os.environ.get("R5_OUTPUT_ROOT", str(DEFAULT_OUTPUT_ROOT)))

    MODEL_NAME = "WasteNet-128K"
    VARIANT_ID = "wastenet_128k"
    ASSISTANT_MODEL_NAME = "Focus-RCNet"
    ASSISTANT_SOURCE_VARIANT = "R2 direct_kd T=4 alpha=0.1"

    STUDENT_IMG_SIZE = 160
    ASSISTANT_IMG_SIZE = 380
    FINAL_EPOCHS = 100
    PILOT_EPOCHS = 60
    SMOKE_EPOCHS = 5
    BATCH_SIZE = 16
    NUM_WORKERS = 0

    MOMENTUM = 0.9
    WEIGHT_DECAY = 1e-4
    SCHEDULER = "CosineAnnealingLR"

    PILOT_GRID_TEMPERATURES = [2.0, 4.0, 6.0]
    PILOT_GRID_ALPHAS = [0.05, 0.1, 0.3]
    PILOT_STAGE2_LRS = [0.001, 0.0005]
    PILOT_ANCHOR = {"temperature": 4.0, "alpha": 0.5}
    PILOT_PRESET = "ta_t4p0_a0p05_lr0p0005"
    PILOT_EARLY_STOPPING = False
    RERUN_TAG = "detpaired_v1"
    SMOKE_PRESET = "ta_t4p0_a0p05_lr0p0005"
    CONFIRMATION_PRESETS = [
        "ta_t4p0_a0p05_lr0p0005",
        "ta_t2p0_a0p3_lr0p0005",
        "ce_finetune_lr0p001",
    ]

    FINAL_KD_TEMPERATURE = 2.0
    FINAL_KD_ALPHA = 0.3
    FINAL_STAGE2_LR = 0.001
    FINAL_USE_KD = True
    FINAL_CONFIG_FROZEN = False

    CHECKPOINT_METRIC = "best_val_accuracy"
    EARLY_STOPPING_MONITOR = "val_loss"
    PATIENCE = 15

    NUM_CLASSES = 6
    CLASS_NAMES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]


def float_tag(value: float) -> str:
    return str(value).replace("-", "m").replace(".", "p")


def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def build_r5_pilot_configs():
    configs = {
        "ce_finetune_lr0p001": {"use_kd": False, "temperature": None, "alpha": None, "stage2_lr": 0.001, "priority": 0},
        "ce_finetune_lr0p0005": {"use_kd": False, "temperature": None, "alpha": None, "stage2_lr": 0.0005, "priority": 1},
    }
    priority = 2
    for lr in [0.001, 0.0005]:
        for temperature in [2.0, 4.0, 6.0]:
            for alpha in [0.05, 0.1, 0.3]:
                preset = f"ta_t{float_tag(temperature)}_a{float_tag(alpha)}_lr{float_tag(lr)}"
                configs[preset] = {
                    "use_kd": True,
                    "temperature": temperature,
                    "alpha": alpha,
                    "stage2_lr": lr,
                    "priority": priority,
                }
                priority += 1
    for lr in [0.001, 0.0005]:
        preset = f"anchor_t4_a0p5_lr{float_tag(lr)}"
        configs[preset] = {
            "use_kd": True,
            "temperature": 4.0,
            "alpha": 0.5,
            "stage2_lr": lr,
            "priority": priority,
        }
        priority += 1
    return configs


cfg = Config()
cfg.SEED = int(os.environ.get("R5_SEED", cfg.SEED))
cfg.WORKFLOW_STAGE = os.environ.get("R5_WORKFLOW_STAGE", cfg.WORKFLOW_STAGE).lower()
cfg.RUN_REPLICATE = os.environ.get("R5_RUN_REPLICATE", cfg.RUN_REPLICATE)
cfg.RUN_PHASE = "final" if cfg.WORKFLOW_STAGE == "final" else "pilot"
cfg.PILOT_CONFIGS = build_r5_pilot_configs()
cfg.PILOT_QUEUE = [name for name, meta in sorted(cfg.PILOT_CONFIGS.items(), key=lambda item: item[1]["priority"])]
cfg.PILOT_PRESET = os.environ.get("R5_PILOT_PRESET", cfg.PILOT_PRESET)
cfg.RERUN_TAG = os.environ.get("R5_RERUN_TAG", cfg.RERUN_TAG)

if cfg.WORKFLOW_STAGE not in {"smoke", "pilot", "final"}:
    raise ValueError(f"Unsupported R5_WORKFLOW_STAGE: {cfg.WORKFLOW_STAGE}")
if cfg.WORKFLOW_STAGE == "final" and not cfg.FINAL_CONFIG_FROZEN:
    raise RuntimeError("R5 final configuration is not frozen. Complete smoke tests and confirmation pilots first.")
if cfg.WORKFLOW_STAGE == "smoke":
    cfg.PILOT_PRESET = cfg.SMOKE_PRESET
elif cfg.WORKFLOW_STAGE == "pilot" and cfg.PILOT_PRESET not in cfg.CONFIRMATION_PRESETS:
    raise ValueError(f"Confirmation pilot must be one of: {cfg.CONFIRMATION_PRESETS}")

cfg.EPOCHS = {"smoke": cfg.SMOKE_EPOCHS, "pilot": cfg.PILOT_EPOCHS, "final": cfg.FINAL_EPOCHS}[cfg.WORKFLOW_STAGE]
if cfg.RUN_PHASE == "pilot":
    if cfg.PILOT_PRESET not in cfg.PILOT_CONFIGS:
        raise ValueError(f"Unknown R5 PILOT_PRESET: {cfg.PILOT_PRESET}. Choose one of: {cfg.PILOT_QUEUE}")
    selected_preset = cfg.PILOT_CONFIGS[cfg.PILOT_PRESET]
    cfg.USE_KD = selected_preset["use_kd"]
    cfg.KD_TEMPERATURE = selected_preset["temperature"]
    cfg.KD_ALPHA = selected_preset["alpha"]
    cfg.STAGE2_LR = selected_preset["stage2_lr"]
else:
    cfg.USE_KD = cfg.FINAL_USE_KD
    cfg.KD_TEMPERATURE = cfg.FINAL_KD_TEMPERATURE
    cfg.KD_ALPHA = cfg.FINAL_KD_ALPHA
    cfg.STAGE2_LR = cfg.FINAL_STAGE2_LR

cfg.TRAINING_MODE = "ta_kd" if cfg.USE_KD else "ce_finetune"
cfg.KD_TYPE = "logits_teacher_assistant" if cfg.USE_KD else "ce_finetune_control"
cfg.EARLY_STOPPING = cfg.RUN_PHASE == "pilot" and cfg.PILOT_EARLY_STOPPING
cfg.EVALUATE_TEST = cfg.RUN_PHASE == "final"

if cfg.RUN_PHASE == "pilot":
    cfg.SETUP_ID = (
        f"r5_{cfg.WORKFLOW_STAGE}_wn128_{cfg.TRAINING_MODE}_{cfg.PILOT_PRESET}_s{cfg.SEED}_"
        f"lr{float_tag(cfg.STAGE2_LR)}_e{cfg.EPOCHS}_{cfg.RERUN_TAG}_"
        f"rep{cfg.RUN_REPLICATE}_"
        f"img{cfg.STUDENT_IMG_SIZE}_assistant{cfg.ASSISTANT_IMG_SIZE}_bs{cfg.BATCH_SIZE}"
    )
    cfg.OUTPUT_DIR = cfg.OUTPUT_ROOT / "pilots" / cfg.EXPERIMENT_ID / cfg.SETUP_ID
else:
    kd_tag = "no-kd" if not cfg.USE_KD else f"t{float_tag(cfg.KD_TEMPERATURE)}_a{float_tag(cfg.KD_ALPHA)}"
    cfg.SETUP_ID = (
        f"r5_final_wn128_{cfg.TRAINING_MODE}_s{cfg.SEED}_{kd_tag}_"
        f"lr{float_tag(cfg.STAGE2_LR)}_e{cfg.EPOCHS}_img{cfg.STUDENT_IMG_SIZE}_bs{cfg.BATCH_SIZE}"
    )
    cfg.OUTPUT_DIR = cfg.OUTPUT_ROOT / "final" / cfg.EXPERIMENT_ID / f"seed_{cfg.SEED}"

cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment : {cfg.EXPERIMENT_NAME}")
print(f"Run phase  : {cfg.RUN_PHASE}")
print(f"Workflow   : {cfg.WORKFLOW_STAGE}")
print(f"Replicate  : {cfg.RUN_REPLICATE}")
print(f"Seed       : {cfg.SEED}")
print(f"Setup ID   : {cfg.SETUP_ID}")
print(f"Preset     : {cfg.PILOT_PRESET if cfg.RUN_PHASE == 'pilot' else 'n/a'}")
print(f"Training   : {cfg.TRAINING_MODE}")
print(f"Use KD     : {cfg.USE_KD}")
print(f"T / alpha  : {cfg.KD_TEMPERATURE} / {cfg.KD_ALPHA}")
print(f"Stage2 LR  : {cfg.STAGE2_LR}")
print(f"Epochs     : {cfg.EPOCHS}")
print(f"Pilot queue ({len(cfg.PILOT_QUEUE)}): {cfg.PILOT_QUEUE}")
print(f"Confirmation queue: {cfg.CONFIRMATION_PRESETS}")
print(f"Rerun tag   : {cfg.RERUN_TAG}")
print(f"Output dir : {cfg.OUTPUT_DIR}")


In [ ]:
# ============================================================
# 3. Reproducibility and Device
# ============================================================

def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True)


seed_everything(cfg.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = cfg.USE_AMP and device.type == "cuda"
print(f"Device: {device}")
print(f"AMP enabled: {AMP_ENABLED}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
    print(f"VRAM GB: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")

## 1. Load R0 Artifacts

Cell ini wajib sukses. Kalau R0 artifact tidak ditemukan, jangan lanjut training.


In [ ]:
# ============================================================
# 4. Resolve R0 Artifacts, R2 Assistant, and R3 Student Init
# ============================================================

def has_r0_artifacts(path: Path) -> bool:
    return (path / "class_mapping.json").exists() and any(path.glob("split_manifest_seed_*.csv"))


def candidate_variants(candidate: Path):
    yield candidate
    yield candidate / "r0_data_protocol"
    yield candidate / "final_research" / "r0_data_protocol"


def discover_r0_dirs():
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]
    discovered = []
    for root in roots:
        if not root.exists():
            continue
        try:
            discovered.extend([path for path in root.rglob("r0_data_protocol") if path.is_dir()])
        except Exception as exc:
            print(f"Skipping R0 discovery under {root}: {exc}")
    return discovered


def resolve_existing_r0_dir(candidates):
    for candidate in candidates:
        candidate = Path(candidate)
        if str(candidate) in {"", "."}:
            continue
        for variant in candidate_variants(candidate):
            if variant.exists() and has_r0_artifacts(variant):
                return variant
    valid_discovered = [path for path in discover_r0_dirs() if has_r0_artifacts(path)]
    if valid_discovered:
        print("Auto-discovered R0 candidates:")
        for path in valid_discovered:
            print(f"- {path}")
        return valid_discovered[0]
    print("Checked R0 candidates:")
    for candidate in candidates:
        print(f"- {candidate}")
    raise FileNotFoundError("R0 data protocol directory not found. Set R0_DATA_PROTOCOL_DIR.")


def resolve_checkpoint(seed: int, roots, filename: str, label: str, extra_subdirs=None) -> Path:
    extra_subdirs = extra_subdirs or []
    candidates = []
    for root in roots:
        root = Path(root)
        if str(root) in {"", "."}:
            continue
        candidates.extend([
            root / f"seed_{seed}" / filename,
            root / filename,
        ])
        for subdir in extra_subdirs:
            subdir = Path(subdir)
            candidates.extend([
                root / subdir / f"seed_{seed}" / filename,
                root / "final_research_kd" / "runs" / "final" / subdir / f"seed_{seed}" / filename,
                root / "runs" / "final" / subdir / f"seed_{seed}" / filename,
            ])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    for root in [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]:
        if root.exists():
            try:
                matches = list(root.rglob(filename))
            except Exception as exc:
                print(f"Skipping checkpoint discovery under {root}: {exc}")
                matches = []
            if matches:
                return matches[0]
    print(f"Checked {label} checkpoint candidates:")
    for candidate in candidates:
        print(f"- {candidate}")
    raise FileNotFoundError(f"{label} checkpoint not found for seed {seed}. Attach output or set the matching env var.")


def resolve_assistant_checkpoint(seed: int) -> Path:
    filename = f"focus_rcnet_teacher_assistant_r2_direct_kd_final_seed_{seed}_best.pth"
    return resolve_checkpoint(seed, cfg.R2_ASSISTANT_ROOT_CANDIDATES, filename, "R2 Focus-RCNet assistant", [Path("R2") / "direct_kd", Path("final") / "R2" / "direct_kd"])


def resolve_student_init_checkpoint(seed: int) -> Path:
    filename = f"wastenet_128k_ce_baseline_r3_final_seed_{seed}_best.pth"
    return resolve_checkpoint(seed, cfg.R3_STUDENT_ROOT_CANDIDATES, filename, "R3 WasteNet-128K CE student init", ["R3", Path("final") / "R3"])


R0_DIR = resolve_existing_r0_dir(cfg.R0_DIR_CANDIDATES)
CLASS_MAPPING_PATH = R0_DIR / "class_mapping.json"
MANIFEST_PATH = R0_DIR / f"split_manifest_seed_{cfg.SEED}.csv"
EXCLUDED_DUPLICATES_PATH = R0_DIR / "excluded_duplicate_conflicts.csv"
STUDENT_INIT_CHECKPOINT_PATH = resolve_student_init_checkpoint(cfg.SEED)
ASSISTANT_CHECKPOINT_PATH = resolve_assistant_checkpoint(cfg.SEED) if cfg.USE_KD else None

required_paths = [CLASS_MAPPING_PATH, MANIFEST_PATH, EXCLUDED_DUPLICATES_PATH, STUDENT_INIT_CHECKPOINT_PATH]
if ASSISTANT_CHECKPOINT_PATH is not None:
    required_paths.append(ASSISTANT_CHECKPOINT_PATH)
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Required artifact missing: {path}")

with CLASS_MAPPING_PATH.open("r", encoding="utf-8") as handle:
    class_mapping = json.load(handle)

manifest_df = pd.read_csv(MANIFEST_PATH)
excluded_df = pd.read_csv(EXCLUDED_DUPLICATES_PATH)
if set(manifest_df["seed"].unique()) != {cfg.SEED}:
    raise ValueError(f"Manifest seed mismatch. Expected only {cfg.SEED}.")

CLASS_NAMES = class_mapping["class_names"]
CLASS_TO_IDX = class_mapping["class_to_idx"]
NUM_CLASSES = len(CLASS_NAMES)

print(f"R0 dir          : {R0_DIR}")
print(f"Manifest        : {MANIFEST_PATH.name}")
print(f"Student init    : {STUDENT_INIT_CHECKPOINT_PATH}")
print(f"Assistant ckpt  : {ASSISTANT_CHECKPOINT_PATH if ASSISTANT_CHECKPOINT_PATH else 'not required for CE fine-tune control'}")
print(f"Rows            : {len(manifest_df)}")
print(f"Classes         : {CLASS_NAMES}")
print(f"Excluded dup    : {len(excluded_df)} rows")
display(manifest_df.groupby(["split", "label", "class_id"], as_index=False).size())


In [ ]:
# ============================================================
# 5. Manifest Validation
# ============================================================

required_columns = {"sample_id", "image_path", "relative_path", "label", "class_id", "split", "seed", "sha256"}
missing_columns = required_columns - set(manifest_df.columns)
if missing_columns:
    raise ValueError(f"Manifest missing columns: {sorted(missing_columns)}")

if set(manifest_df["seed"].unique()) != {cfg.SEED}:
    raise ValueError(f"Manifest seed mismatch. Expected only {cfg.SEED}.")

if set(manifest_df["split"].unique()) != {"train", "val", "test"}:
    raise ValueError("Manifest must contain train, val, and test splits.")

excluded_ids = set(excluded_df.get("sample_id", []))
if excluded_ids & set(manifest_df["sample_id"]):
    raise AssertionError("Excluded duplicate-conflict samples are present in this manifest.")

for class_name, class_id in CLASS_TO_IDX.items():
    rows = manifest_df[manifest_df["label"] == class_name]
    if rows.empty:
        raise AssertionError(f"Missing class in manifest: {class_name}")
    if set(rows["class_id"].unique()) != {class_id}:
        raise AssertionError(f"Class id mismatch for {class_name}")

print("Manifest validation passed")

## 2. Dataset and DataLoader


In [ ]:
# ============================================================
# 6. Dual-Resolution Transforms
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def coarse_dropout(img_size: int):
    min_h = int(img_size * 0.05)
    max_h = int(img_size * 0.20)
    try:
        return A.CoarseDropout(
            num_holes_range=(1, 1),
            hole_height_range=(min_h, max_h),
            hole_width_range=(min_h, max_h),
            fill=0,
            p=0.5,
        )
    except TypeError:
        return A.CoarseDropout(
            max_holes=1,
            min_height=min_h,
            max_height=max_h,
            min_width=min_h,
            max_width=max_h,
            fill_value=0,
            p=0.5,
        )


def seeded_compose(transforms, seed: int):
    transform = A.Compose(transforms)
    if hasattr(transform, "set_random_seed"):
        transform.set_random_seed(seed)
    return transform


def shared_train_transform(img_size: int, seed: int):
    return seeded_compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        coarse_dropout(img_size),
    ], seed)


def model_view_transform(img_size: int):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def eval_transform_for(img_size: int):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


train_shared_transform = shared_train_transform(cfg.ASSISTANT_IMG_SIZE, cfg.SEED)
student_view_transform = model_view_transform(cfg.STUDENT_IMG_SIZE)
assistant_view_transform = model_view_transform(cfg.ASSISTANT_IMG_SIZE)
student_eval_transform = eval_transform_for(cfg.STUDENT_IMG_SIZE)
assistant_eval_transform = eval_transform_for(cfg.ASSISTANT_IMG_SIZE)

print("Paired dual-resolution transforms ready")
print("One shared random augmentation is applied before both model views are created.")
print(f"Student image   : {cfg.STUDENT_IMG_SIZE}x{cfg.STUDENT_IMG_SIZE}")
print(f"Assistant image : {cfg.ASSISTANT_IMG_SIZE}x{cfg.ASSISTANT_IMG_SIZE}")


In [ ]:
# ============================================================
# 7. Manifest Dataset
# ============================================================

class DualResolutionTrashNetDataset(Dataset):
    def __init__(self, df: pd.DataFrame, dataset_dir: Path, shared_transform=None, student_transform=None, assistant_transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.dataset_dir = Path(dataset_dir)
        self.shared_transform = shared_transform
        self.student_transform = student_transform
        self.assistant_transform = assistant_transform

    def __len__(self):
        return len(self.df)

    def resolve_path(self, row) -> Path:
        absolute_path = Path(row["image_path"])
        if absolute_path.exists():
            return absolute_path
        fallback_path = self.dataset_dir / row["relative_path"]
        if fallback_path.exists():
            return fallback_path
        raise FileNotFoundError(f"Image not found: {absolute_path} or {fallback_path}")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = self.resolve_path(row)
        image = np.array(Image.open(image_path).convert("RGB"))
        paired_image = self.shared_transform(image=image)["image"] if self.shared_transform else image
        student_image = self.student_transform(image=paired_image)["image"] if self.student_transform else paired_image
        assistant_image = self.assistant_transform(image=paired_image)["image"] if self.assistant_transform else student_image
        return {
            "student_image": student_image,
            "assistant_image": assistant_image,
            "label": int(row["class_id"]),
            "sample_id": row["sample_id"],
            "relative_path": row["relative_path"],
        }


train_df = manifest_df[manifest_df["split"] == "train"].copy()
val_df = manifest_df[manifest_df["split"] == "val"].copy()
test_df = manifest_df[manifest_df["split"] == "test"].copy()

train_dataset = DualResolutionTrashNetDataset(
    train_df,
    cfg.DATASET_DIR,
    shared_transform=train_shared_transform,
    student_transform=student_view_transform,
    assistant_transform=assistant_view_transform if cfg.USE_KD else None,
)
val_dataset = DualResolutionTrashNetDataset(
    val_df,
    cfg.DATASET_DIR,
    student_transform=student_eval_transform,
    assistant_transform=assistant_eval_transform if cfg.USE_KD else None,
)
test_dataset = DualResolutionTrashNetDataset(
    test_df,
    cfg.DATASET_DIR,
    student_transform=student_eval_transform,
    assistant_transform=assistant_eval_transform if cfg.USE_KD else None,
)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")


In [ ]:
# ============================================================
# 8. DataLoaders and Distribution
# ============================================================

def worker_init_fn(worker_id):
    worker_seed = cfg.SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


generator = torch.Generator()
generator.manual_seed(cfg.SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
    generator=generator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
)

split_counts = manifest_df.groupby(["split", "label", "class_id"], as_index=False).size()
display(split_counts)
print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")

## 3. Student, Assistant, and Training Components


In [ ]:
# ============================================================
# 9. WasteNet-128K Student and Focus-RCNet Assistant
# ============================================================

@dataclass(frozen=True)
class WasteNetVariant:
    variant_id: str
    display_name: str
    channels: tuple
    repeats: tuple
    expand_ratio: float
    classifier_hidden: int
    expected_params: int
    checkpoint_prefix: str


WASTENET_128K = WasteNetVariant(
    variant_id="wastenet_128k",
    display_name="WasteNet-128K",
    channels=(16, 32, 64, 128, 192),
    repeats=(1, 1, 1, 1),
    expand_ratio=2.0,
    classifier_hidden=32,
    expected_params=128_086,
    checkpoint_prefix="wastenet_128k",
)


class ConvBNAct(nn.Sequential):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, groups: int = 1):
        super().__init__(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=kernel_size // 2,
                groups=groups,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )


class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, expand_ratio: float = 1.0):
        super().__init__()
        hidden_channels = int(round(in_channels * expand_ratio))
        layers = []
        if hidden_channels != in_channels:
            layers.append(ConvBNAct(in_channels, hidden_channels, kernel_size=1, stride=1))
        layers.extend([
            ConvBNAct(hidden_channels, hidden_channels, kernel_size=3, stride=stride, groups=hidden_channels),
            nn.Conv2d(hidden_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        ])
        self.block = nn.Sequential(*layers)
        self.activation = nn.SiLU(inplace=True)
        self.use_residual = stride == 1 and in_channels == out_channels

    def forward(self, x):
        y = self.block(x)
        if self.use_residual:
            y = y + x
        return self.activation(y)


class WasteNet(nn.Module):
    def __init__(self, variant: WasteNetVariant, num_classes: int = 6, dropout: float = 0.2):
        super().__init__()
        layers = [ConvBNAct(3, variant.channels[0], kernel_size=3, stride=2)]
        in_channels = variant.channels[0]

        for out_channels, repeat in zip(variant.channels[1:], variant.repeats):
            for block_idx in range(repeat):
                stride = 2 if block_idx == 0 else 1
                layers.append(
                    DepthwiseSeparableBlock(
                        in_channels,
                        out_channels,
                        stride=stride,
                        expand_ratio=variant.expand_ratio,
                    )
                )
                in_channels = out_channels

        self.features = nn.Sequential(*layers)
        if variant.classifier_hidden > 0:
            self.classifier = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Dropout(dropout),
                nn.Linear(in_channels, variant.classifier_hidden),
                nn.SiLU(inplace=True),
                nn.Dropout(dropout),
                nn.Linear(variant.classifier_hidden, num_classes),
            )
        else:
            self.classifier = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Dropout(dropout),
                nn.Linear(in_channels, num_classes),
            )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def create_wastenet_128k() -> WasteNet:
    return WasteNet(variant=WASTENET_128K, num_classes=NUM_CLASSES)


# ============================================================
# Focus-RCNet Assistant
# ============================================================


class Focus(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels * 4, out_channels, kernel_size, stride=1, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(torch.cat([
            x[..., ::2, ::2],
            x[..., 1::2, ::2],
            x[..., ::2, 1::2],
            x[..., 1::2, 1::2],
        ], dim=1))


class SimAM(nn.Module):
    def __init__(self, e_lambda: float = 1e-4):
        super().__init__()
        self.e_lambda = e_lambda

    def forward(self, x):
        _, _, height, width = x.size()
        n = height * width - 1
        x_minus_mu_sq = (x - x.mean(dim=[2, 3], keepdim=True)).pow(2)
        y = x_minus_mu_sq / (
            4 * (x_minus_mu_sq.sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)
        ) + 0.5
        return x * torch.sigmoid(y)


class SandglassBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, reduction: int = 2):
        super().__init__()
        self.use_residual = stride == 1 and in_channels == out_channels
        mid_channels = max(in_channels // reduction, 1)
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, stride=stride, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(in_channels, mid_channels, 1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.Conv2d(mid_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, groups=out_channels, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        output = self.layers(x)
        return x + output if self.use_residual else output


class FocusRCNet(nn.Module):
    STAGE_CONFIG = [(48, 4, 2, 2), (96, 3, 2, 2), (192, 2, 2, 2), (384, 2, 2, 2)]

    def __init__(self, num_classes: int = 6, dropout: float = 0.2):
        super().__init__()
        self.focus = Focus(3, 24, kernel_size=1)
        stages = []
        in_channels = 24
        for out_channels, num_blocks, stride, reduction in self.STAGE_CONFIG:
            blocks = []
            for block_idx in range(num_blocks):
                block_stride = stride if block_idx == 0 else 1
                blocks.append(SandglassBlock(in_channels, out_channels, stride=block_stride, reduction=reduction))
                in_channels = out_channels
            blocks.append(SimAM())
            stages.append(nn.Sequential(*blocks))
        self.stages = nn.Sequential(*stages)
        self.conv5 = nn.Sequential(
            nn.Conv2d(in_channels, 512, kernel_size=1, bias=False),
            nn.BatchNorm2d(512),
            nn.SiLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes),
        )
        self._initialize_weights()

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, 0, 0.01)
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.focus(x)
        x = self.stages(x)
        x = self.conv5(x)
        return self.classifier(x)


def create_focus_rcnet(num_classes: int = 6):
    return FocusRCNet(num_classes=num_classes)



def load_student_initialization(model: nn.Module, checkpoint_path: Path):
    loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model.load_state_dict(loaded["model_state_dict"])
    print(f"Student init loaded: {checkpoint_path}")
    print(f"R3 best epoch      : {loaded.get('best_epoch', 'n/a')}")
    print(f"R3 best val acc    : {loaded.get('best_val_acc', 'n/a')}")
    return loaded


def load_assistant_model(checkpoint_path: Path):
    loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    assistant = create_focus_rcnet(NUM_CLASSES)
    assistant.load_state_dict(loaded["model_state_dict"])
    assistant = assistant.to(device).eval()
    for parameter in assistant.parameters():
        parameter.requires_grad = False
    print(f"Assistant loaded   : {checkpoint_path}")
    print(f"R2 variant         : {loaded.get('r2_variant', 'direct_kd')}")
    print(f"R2 best epoch      : {loaded.get('best_epoch', 'n/a')}")
    print(f"R2 best val acc    : {loaded.get('best_val_acc', 'n/a')}")
    return assistant, loaded


model = create_wastenet_128k().to(device)
student_init_checkpoint = load_student_initialization(model, STUDENT_INIT_CHECKPOINT_PATH)
assistant_model = None
assistant_checkpoint = None
if cfg.USE_KD:
    assistant_model, assistant_checkpoint = load_assistant_model(ASSISTANT_CHECKPOINT_PATH)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

sample = torch.randn(1, 3, cfg.STUDENT_IMG_SIZE, cfg.STUDENT_IMG_SIZE).to(device)
with torch.no_grad():
    sample_output = model(sample)
assert tuple(sample_output.shape) == (1, NUM_CLASSES), f"Unexpected output shape: {sample_output.shape}"
assert total_params == WASTENET_128K.expected_params, f"Expected 128,086 params, got {total_params:,}"

if cfg.USE_KD:
    assistant_sample = torch.randn(1, 3, cfg.ASSISTANT_IMG_SIZE, cfg.ASSISTANT_IMG_SIZE).to(device)
    with torch.no_grad():
        assistant_output = assistant_model(assistant_sample)
    assert tuple(assistant_output.shape) == (1, NUM_CLASSES), f"Unexpected assistant output shape: {assistant_output.shape}"
    del assistant_sample, assistant_output

print(f"Student model : {cfg.MODEL_NAME}")
print(f"Assistant     : {cfg.ASSISTANT_MODEL_NAME if cfg.USE_KD else 'not used for this preset'}")
print(f"Variant       : {WASTENET_128K.variant_id}")
print(f"Total params  : {total_params:,}")
print(f"Trainable     : {trainable_params:,}")
print(f"Output shape  : {list(sample_output.shape)}")

del sample, sample_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# 10. Loss, Optimizer, and Scheduler
# ============================================================

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=cfg.STAGE2_LR, momentum=cfg.MOMENTUM, weight_decay=cfg.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)
scaler = GradScaler(enabled=AMP_ENABLED)


def kd_loss(student_logits, assistant_logits, labels):
    temperature = cfg.KD_TEMPERATURE
    alpha = cfg.KD_ALPHA
    student_log_soft = F.log_softmax(student_logits.float() / temperature, dim=1)
    assistant_soft = F.softmax(assistant_logits.float() / temperature, dim=1)
    loss_soft = F.kl_div(student_log_soft, assistant_soft, reduction="batchmean") * (temperature ** 2)
    loss_hard = criterion(student_logits.float(), labels)
    loss = alpha * loss_soft + (1.0 - alpha) * loss_hard
    return loss, loss_soft, loss_hard

print("Training components ready")
print(f"Optimizer LR: {cfg.STAGE2_LR}")
print(f"KD enabled  : {cfg.USE_KD}")


In [ ]:
# ============================================================
# 11. Train / Validate Helpers
# ============================================================

def run_one_train_epoch(student, assistant, loader):
    student.train()
    if assistant is not None:
        assistant.eval()
    total_loss = 0.0
    total_soft_loss = 0.0
    total_hard_loss = 0.0
    total_correct = 0
    total = 0
    for batch in loader:
        student_images = batch["student_image"].to(device, non_blocking=True)
        assistant_images = batch["assistant_image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=AMP_ENABLED):
            student_logits = student(student_images)
            if cfg.USE_KD:
                with torch.no_grad():
                    assistant_logits = assistant(assistant_images)
        with autocast(enabled=False):
            if cfg.USE_KD:
                loss, loss_soft, loss_hard = kd_loss(student_logits, assistant_logits, labels)
                soft_value = loss_soft.item()
                hard_value = loss_hard.item()
            else:
                loss = criterion(student_logits.float(), labels)
                soft_value = float("nan")
                hard_value = loss.item()
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite training loss detected")
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_soft_loss += soft_value * batch_size
        total_hard_loss += hard_value * batch_size
        total_correct += (student_logits.argmax(dim=1) == labels).sum().item()
        total += batch_size
    return total_loss / total, total_correct / total, total_soft_loss / total, total_hard_loss / total


@torch.no_grad()
def run_eval_epoch(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for batch in loader:
        images = batch["student_image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        with autocast(enabled=AMP_ENABLED):
            logits = model(images)
            loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, total_correct / total


## 4. Training


In [ ]:
# ============================================================
# 12. Main Training Loop
# ============================================================

history = []
best_val_acc = -1.0
best_epoch = 0
best_model_state = None
best_monitor_value = np.inf if cfg.EARLY_STOPPING_MONITOR == "val_loss" else -np.inf
epochs_without_improvement = 0
total_start = time.time()

print(f"Starting {cfg.EXPERIMENT_NAME} | seed={cfg.SEED} | preset={cfg.PILOT_PRESET if cfg.RUN_PHASE == 'pilot' else 'final'} | mode={cfg.TRAINING_MODE} | T={cfg.KD_TEMPERATURE} | alpha={cfg.KD_ALPHA} | lr={cfg.STAGE2_LR}")
for epoch in range(1, cfg.EPOCHS + 1):
    epoch_start = time.time()
    current_lr = optimizer.param_groups[0]["lr"]
    train_loss, train_acc, train_soft_loss, train_hard_loss = run_one_train_epoch(model, assistant_model, train_loader)
    val_loss, val_acc = run_eval_epoch(model, val_loader)
    scheduler.step()
    epoch_time = time.time() - epoch_start
    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())
    if cfg.EARLY_STOPPING_MONITOR == "val_loss":
        monitor_value = val_loss
        monitor_improved = monitor_value < best_monitor_value - 1e-8
    elif cfg.EARLY_STOPPING_MONITOR == "val_acc":
        monitor_value = val_acc
        monitor_improved = monitor_value > best_monitor_value + 1e-8
    else:
        raise ValueError(f"Unsupported EARLY_STOPPING_MONITOR: {cfg.EARLY_STOPPING_MONITOR}")
    if monitor_improved:
        best_monitor_value = monitor_value
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_loss_soft": train_soft_loss,
        "train_loss_hard": train_hard_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "lr": current_lr,
        "epoch_time_sec": epoch_time,
        "is_best": is_best,
        "early_stop_monitor": cfg.EARLY_STOPPING_MONITOR,
        "early_stop_value": monitor_value,
        "early_stop_improved": monitor_improved,
        "epochs_without_improvement": epochs_without_improvement,
    })
    marker = " BEST" if is_best else ""
    early_stop_status = f" | es_wait={epochs_without_improvement}/{cfg.PATIENCE}" if cfg.EARLY_STOPPING else ""
    print(
        f"Epoch {epoch:03d}/{cfg.EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"soft={train_soft_loss:.4f} hard={train_hard_loss:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"lr={current_lr:.6f} time={epoch_time:.1f}s{early_stop_status}{marker}"
    )
    if cfg.EARLY_STOPPING and epochs_without_improvement >= cfg.PATIENCE:
        print(f"Early stopping triggered at epoch {epoch} ({cfg.EARLY_STOPPING_MONITOR} did not improve for {cfg.PATIENCE} epochs).")
        break

total_train_time = time.time() - total_start
history_df = pd.DataFrame(history)
if best_model_state is None:
    raise RuntimeError("No best model state was captured.")
print("Training complete")
print(f"Best epoch: {best_epoch}")
print(f"Best val acc: {best_val_acc:.6f}")
print(f"Total minutes: {total_train_time / 60:.1f}")


## 5. Evaluation and Prediction CSV


In [ ]:
# ============================================================
# 13. Evaluation Helpers
# ============================================================

@torch.no_grad()
def collect_predictions(model, loader, split_name: str) -> pd.DataFrame:
    model.eval()
    rows = []
    for batch in loader:
        images = batch["student_image"].to(device, non_blocking=True)
        labels = batch["label"].cpu().numpy()
        with autocast(enabled=AMP_ENABLED):
            logits = model(images)
            probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

        for i in range(len(labels)):
            row = {
                "sample_id": batch["sample_id"][i],
                "image_path": batch["relative_path"][i],
                "label": CLASS_NAMES[int(labels[i])],
                "label_id": int(labels[i]),
                "prediction": CLASS_NAMES[int(preds[i])],
                "prediction_id": int(preds[i]),
                "seed": cfg.SEED,
                "model_id": cfg.EXPERIMENT_ID,
                "variant": cfg.TRAINING_MODE,
                "split": split_name,
            }
            for class_idx, class_name in enumerate(CLASS_NAMES):
                row[f"prob_{class_name}"] = float(probs[i, class_idx])
            rows.append(row)
    return pd.DataFrame(rows)


def compute_metrics(pred_df: pd.DataFrame) -> dict:
    y_true = pred_df["label_id"].to_numpy()
    y_pred = pred_df["prediction_id"].to_numpy()
    prob_cols = [f"prob_{class_name}" for class_name in CLASS_NAMES]
    y_prob = pred_df[prob_cols].to_numpy()

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    try:
        auc_macro_ovr = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except ValueError:
        auc_macro_ovr = np.nan

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "auc_macro_ovr": float(auc_macro_ovr),
    }


def save_confusion_matrix(pred_df: pd.DataFrame, split_name: str):
    cm = confusion_matrix(pred_df["label_id"], pred_df["prediction_id"], labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{cfg.EXPERIMENT_ID} WasteNet-128K Direct KD - {split_name}")
    for r in range(NUM_CLASSES):
        for c in range(NUM_CLASSES):
            ax.text(c, r, str(cm[r, c]), ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    path = cfg.OUTPUT_DIR / f"confusion_matrix_{split_name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path


In [ ]:
# ============================================================
# 14. Load Best State and Evaluate
# ============================================================

model.load_state_dict(best_model_state)

val_predictions_df = collect_predictions(model, val_loader, "val")
val_metrics = compute_metrics(val_predictions_df)

test_predictions_df = pd.DataFrame()
test_metrics = None
if cfg.EVALUATE_TEST and cfg.RUN_PHASE == "final":
    test_predictions_df = collect_predictions(model, test_loader, "test")
    test_metrics = compute_metrics(test_predictions_df)
else:
    print("Independent test evaluation skipped. Set RUN_PHASE='final' and EVALUATE_TEST=True to enable it.")

print("Validation metrics:")
display(pd.DataFrame([val_metrics]))
if test_metrics is not None:
    print("Test metrics:")
    display(pd.DataFrame([test_metrics]))

val_pred_path = cfg.OUTPUT_DIR / f"predictions_{cfg.EXPERIMENT_ID.lower()}_{cfg.TRAINING_MODE}_{cfg.RUN_PHASE}_seed_{cfg.SEED}_val.csv"
val_predictions_df.to_csv(val_pred_path, index=False)
print(f"Saved: {val_pred_path}")

test_pred_path = None
if not test_predictions_df.empty:
    test_pred_path = cfg.OUTPUT_DIR / f"predictions_{cfg.EXPERIMENT_ID.lower()}_{cfg.TRAINING_MODE}_{cfg.RUN_PHASE}_seed_{cfg.SEED}_test.csv"
    test_predictions_df.to_csv(test_pred_path, index=False)
    print(f"Saved: {test_pred_path}")

val_cm_path = save_confusion_matrix(val_predictions_df, "val")
test_cm_path = None
if not test_predictions_df.empty:
    test_cm_path = save_confusion_matrix(test_predictions_df, "test")


## 6. Save Artifacts


In [ ]:
# ============================================================
# 15. Training Curves and History
# ============================================================

prefix = f"{cfg.EXPERIMENT_ID.lower()}_{cfg.TRAINING_MODE}_{cfg.RUN_PHASE}_seed_{cfg.SEED}"
history_path = cfg.OUTPUT_DIR / f"training_history_{prefix}.csv"
history_df.to_csv(history_path, index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val")
axes[0].axvline(best_epoch, color="red", linestyle="--", alpha=0.6)
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(history_df["epoch"], history_df["train_acc"], label="train")
axes[1].plot(history_df["epoch"], history_df["val_acc"], label="val")
axes[1].axvline(best_epoch, color="red", linestyle="--", alpha=0.6)
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[2].plot(history_df["epoch"], history_df["lr"], color="green", label="lr")
axes[2].set_title("Learning Rate")
axes[2].set_xlabel("Epoch")
axes[2].grid(alpha=0.3)
fig.suptitle(f"{cfg.EXPERIMENT_ID} WasteNet-128K R5 Teacher-Assistant KD - seed {cfg.SEED} | mode={cfg.TRAINING_MODE}, T={cfg.KD_TEMPERATURE}, alpha={cfg.KD_ALPHA}, lr={cfg.STAGE2_LR}")
fig.tight_layout()
curve_path = cfg.OUTPUT_DIR / f"training_curves_{prefix}.png"
fig.savefig(curve_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {history_path}")
print(f"Saved: {curve_path}")


In [ ]:
# ============================================================
# 16. Checkpoint, Metrics, Artifact Manifest
# ============================================================

metrics_rows = [{"split": "val", **val_metrics}]
if test_metrics is not None:
    metrics_rows.append({"split": "test", **test_metrics})
metrics_df = pd.DataFrame(metrics_rows)
metrics_path = cfg.OUTPUT_DIR / f"metrics_{prefix}.csv"
metrics_df.to_csv(metrics_path, index=False)

repro_history_columns = [column for column in history_df.columns if column != "epoch_time_sec"]
history_payload = history_df[repro_history_columns].to_csv(index=False, float_format="%.17g", lineterminator="\n")
prediction_payload = val_predictions_df.to_csv(index=False, float_format="%.17g", lineterminator="\n")
reproducibility_fingerprint = {
    "history_columns": repro_history_columns,
    "history_sha256_excluding_epoch_time": hashlib.sha256(history_payload.encode("utf-8")).hexdigest(),
    "val_predictions_sha256": hashlib.sha256(prediction_payload.encode("utf-8")).hexdigest(),
}
fingerprint_path = cfg.OUTPUT_DIR / f"reproducibility_fingerprint_{prefix}.json"
fingerprint_path.write_text(json.dumps(reproducibility_fingerprint, indent=2), encoding="utf-8")

config_dict = {
    "experiment_id": cfg.EXPERIMENT_ID,
    "experiment_name": cfg.EXPERIMENT_NAME,
    "seed": cfg.SEED,
    "run_phase": cfg.RUN_PHASE,
    "workflow_stage": cfg.WORKFLOW_STAGE,
    "run_replicate": cfg.RUN_REPLICATE,
    "setup_id": cfg.SETUP_ID,
    "pilot_preset": cfg.PILOT_PRESET if cfg.RUN_PHASE == "pilot" else None,
    "rerun_tag": cfg.RERUN_TAG if cfg.RUN_PHASE == "pilot" else None,
    "smoke_preset": cfg.SMOKE_PRESET,
    "confirmation_presets": cfg.CONFIRMATION_PRESETS,
    "pilot_early_stopping": cfg.PILOT_EARLY_STOPPING,
    "output_root": str(cfg.OUTPUT_ROOT),
    "output_dir": str(cfg.OUTPUT_DIR),
    "dataset_dir": str(cfg.DATASET_DIR),
    "r0_dir": str(R0_DIR),
    "manifest_path": str(MANIFEST_PATH),
    "student_init_checkpoint_path": str(STUDENT_INIT_CHECKPOINT_PATH),
    "student_init_checkpoint_sha256": file_sha256(STUDENT_INIT_CHECKPOINT_PATH),
    "assistant_checkpoint_path": str(ASSISTANT_CHECKPOINT_PATH) if ASSISTANT_CHECKPOINT_PATH else None,
    "assistant_checkpoint_sha256": file_sha256(ASSISTANT_CHECKPOINT_PATH) if ASSISTANT_CHECKPOINT_PATH else None,
    "split_manifest_sha256": file_sha256(MANIFEST_PATH),
    "assistant_model_name": cfg.ASSISTANT_MODEL_NAME,
    "assistant_source_variant": cfg.ASSISTANT_SOURCE_VARIANT,
    "model_name": cfg.MODEL_NAME,
    "variant_id": WASTENET_128K.variant_id,
    "variant": {
        "channels": WASTENET_128K.channels,
        "repeats": WASTENET_128K.repeats,
        "expand_ratio": WASTENET_128K.expand_ratio,
        "classifier_hidden": WASTENET_128K.classifier_hidden,
        "expected_params": WASTENET_128K.expected_params,
    },
    "training_mode": cfg.TRAINING_MODE,
    "knowledge_distillation": bool(cfg.USE_KD),
    "kd_type": cfg.KD_TYPE,
    "kd_temperature": cfg.KD_TEMPERATURE,
    "kd_alpha": cfg.KD_ALPHA,
    "stage2_lr": cfg.STAGE2_LR,
    "pilot_grid_temperatures": cfg.PILOT_GRID_TEMPERATURES,
    "pilot_grid_alphas": cfg.PILOT_GRID_ALPHAS,
    "pilot_stage2_lrs": cfg.PILOT_STAGE2_LRS,
    "pilot_anchor": cfg.PILOT_ANCHOR,
    "pilot_configs": cfg.PILOT_CONFIGS,
    "pilot_queue": cfg.PILOT_QUEUE,
    "student_img_size": cfg.STUDENT_IMG_SIZE,
    "assistant_img_size": cfg.ASSISTANT_IMG_SIZE,
    "epochs": cfg.EPOCHS,
    "final_epochs": cfg.FINAL_EPOCHS,
    "pilot_epochs": cfg.PILOT_EPOCHS,
    "batch_size": cfg.BATCH_SIZE,
    "optimizer": "SGD",
    "lr": cfg.STAGE2_LR,
    "momentum": cfg.MOMENTUM,
    "weight_decay": cfg.WEIGHT_DECAY,
    "scheduler": cfg.SCHEDULER,
    "use_amp": AMP_ENABLED,
    "deterministic": cfg.DETERMINISTIC,
    "num_workers": cfg.NUM_WORKERS,
    "paired_augmentation": True,
    "paired_augmentation_base_size": cfg.ASSISTANT_IMG_SIZE,
    "early_stopping": cfg.EARLY_STOPPING,
    "evaluate_test": cfg.EVALUATE_TEST,
    "early_stopping_monitor": cfg.EARLY_STOPPING_MONITOR,
    "patience": cfg.PATIENCE,
    "checkpoint_metric": cfg.CHECKPOINT_METRIC,
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "train_size": int(len(train_df)),
    "val_size": int(len(val_df)),
    "test_size": int(len(test_df)),
    "total_params": int(total_params),
    "trainable_params": int(trainable_params),
    "total_train_time_sec": float(total_train_time),
}
checkpoint = {
    "model_state_dict": best_model_state,
    "student_init_checkpoint_path": str(STUDENT_INIT_CHECKPOINT_PATH),
    "assistant_checkpoint_path": str(ASSISTANT_CHECKPOINT_PATH) if ASSISTANT_CHECKPOINT_PATH else None,
    "best_epoch": best_epoch,
    "best_val_acc": float(best_val_acc),
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
    "config": config_dict,
    "class_mapping": class_mapping,
}
checkpoint_path = cfg.OUTPUT_DIR / f"wastenet_128k_teacher_assistant_kd_{prefix}_best.pth"
torch.save(checkpoint, checkpoint_path)
config_path = cfg.OUTPUT_DIR / f"config_{prefix}.json"
config_path.write_text(json.dumps(config_dict, indent=2), encoding="utf-8")
artifact_manifest = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment_id": cfg.EXPERIMENT_ID,
    "seed": cfg.SEED,
    "run_phase": cfg.RUN_PHASE,
    "workflow_stage": cfg.WORKFLOW_STAGE,
    "run_replicate": cfg.RUN_REPLICATE,
    "training_mode": cfg.TRAINING_MODE,
    "pilot_preset": cfg.PILOT_PRESET if cfg.RUN_PHASE == "pilot" else None,
    "rerun_tag": cfg.RERUN_TAG if cfg.RUN_PHASE == "pilot" else None,
    "kd_temperature": cfg.KD_TEMPERATURE,
    "kd_alpha": cfg.KD_ALPHA,
    "stage2_lr": cfg.STAGE2_LR,
    "setup_id": cfg.SETUP_ID,
    "output_dir": str(cfg.OUTPUT_DIR),
    "artifacts": {
        "checkpoint": str(checkpoint_path),
        "config": str(config_path),
        "history": str(history_path),
        "metrics": str(metrics_path),
        "val_predictions": str(val_pred_path),
        "test_predictions": str(test_pred_path) if test_pred_path else None,
        "val_confusion_matrix": str(val_cm_path),
        "test_confusion_matrix": str(test_cm_path) if test_cm_path else None,
        "training_curves": str(curve_path),
        "reproducibility_fingerprint": str(fingerprint_path),
    },
}
artifact_manifest_path = cfg.OUTPUT_DIR / f"artifact_manifest_{prefix}.json"
artifact_manifest_path.write_text(json.dumps(artifact_manifest, indent=2), encoding="utf-8")
print(f"Saved checkpoint: {checkpoint_path}")
print(f"Saved metrics   : {metrics_path}")
print(f"Saved manifest  : {artifact_manifest_path}")
print(f"History fingerprint: {reproducibility_fingerprint['history_sha256_excluding_epoch_time']}")
print(f"Prediction fingerprint: {reproducibility_fingerprint['val_predictions_sha256']}")
display(metrics_df)


In [ ]:
# ============================================================
# 17. Checkpoint Verification
# ============================================================

loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
verify_model = create_wastenet_128k()
verify_model.load_state_dict(loaded["model_state_dict"])

print("Checkpoint verification passed")
print(f"Best epoch   : {loaded['best_epoch']}")
print(f"Best val acc : {loaded['best_val_acc']:.6f}")

del loaded, verify_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Output Artifacts

Per seed/config, notebook ini menghasilkan checkpoint WasteNet-128K R5, history CSV, metrics CSV, prediction CSV, confusion matrix, training curves, config JSON, dan artifact manifest. Pada pilot, gunakan validation metric untuk memilih satu konfigurasi R5. Pada final, jalankan 5 seed dengan konfigurasi yang sudah dibekukan dan baru evaluasi independent test set.


In [ ]:
print("R5 complete")
print(f"Seed: {cfg.SEED}")
print(f"Pilot preset: {cfg.PILOT_PRESET if cfg.RUN_PHASE == 'pilot' else 'n/a'}")
print(f"Training mode: {cfg.TRAINING_MODE}")
print(f"Use KD: {cfg.USE_KD}")
print(f"T / alpha: {cfg.KD_TEMPERATURE} / {cfg.KD_ALPHA}")
print(f"Stage2 LR: {cfg.STAGE2_LR}")
print(f"Best epoch: {best_epoch}")
print(f"Best validation accuracy: {best_val_acc:.6f}")
print(f"Output directory: {cfg.OUTPUT_DIR}")
